In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Calibri'] + plt.rcParams['font.sans-serif']
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['axes.labelcolor'] = '#222222'
plt.rcParams['text.color'] = '#222222'
plt.rcParams['xtick.color'] = '#333333'
plt.rcParams['ytick.color'] = '#333333'

VALUE_COL = 'avg_saliency' 
VALUE_SCALE = 1e-6 

EXCLUDE_AREAS = ['Medial_wall', 'Cerebral Cortex', 'Inferior Lateral Ventricle', 'Lateral Ventricle']

FOLD_SITE_NAMES = {0: 'KKI', 1: 'NYU', 2: 'OHSU', 3: 'SDSU', 4: 'TCD'} 

saliency = pd.read_csv('/path/to/avg_ig_by_region.csv')
lookup = pd.read_csv('/path/to/fs_lookup.csv', usecols=['code', 'region'])
lobe_info = pd.read_csv('/path/to/destrieux - lobe.csv')
saliency_sub = pd.read_csv('/path/to/ig_correct_fold/avg_ig_by_region_sub.csv')

idx_to_region = {
    2: "Cerebral White Matter (L)", 41: "Cerebral White Matter (R)",
    3: "Cerebral Cortex (L)", 42: "Cerebral Cortex (R)",
    4: "Lateral Ventricle (L)", 43: "Lateral Ventricle (R)",
    5: "Inferior Lateral Ventricle (L)", 44: "Inferior Lateral Ventricle (R)",
    7: "Cerebellum White Matter (L)", 46: "Cerebellum White Matter (R)",
    8: "Cerebellum Cortex (L)", 47: "Cerebellum Cortex (R)",
    10: "Thalamus (L)", 49: "Thalamus (R)",
    11: "Caudate (L)", 50: "Caudate (R)",
    12: "Putamen (L)", 51: "Putamen (R)",
    13: "Pallidum (L)", 52: "Pallidum (R)",
    17: "Hippocampus (L)", 53: "Hippocampus (R)",
    18: "Amygdala (L)", 54: "Amygdala (R)",
    26: "Accumbens Area (L)", 58: "Accumbens Area (R)",
}
hemi_letter_map = {'L': 'lh', 'R': 'rh'}
lobe_order = ['frontal', 'temporal', 'parietal', 'occipital', 'limbic', 'insular',
              'subcortical', 'unknown']

def sci_fmt(x, pos):
    if x == 0:
        return '0'
    mantissa, exp = f'{x:.2e}'.split('e')
    mantissa = mantissa.rstrip('0').rstrip('.')
    return f'{mantissa}e{int(exp)}'


def avg_by_fold(df, value_col):
    return (
        df.groupby(['area', 'hemi', 'fold'], observed=True)[value_col]
        .mean().reset_index().rename(columns={value_col: 'avg'})
    )


merged = pd.merge(saliency, lookup, left_on='region_id', right_on='code', how='left')
merged = merged.dropna(subset=['region'])
merged = merged[~merged['region'].str.contains('Unknown', case=False)]
merged['hemi'] = merged['region'].str.extract(r'ctx_(lh|rh)_')
merged['area'] = merged['region'].str.replace(r'^ctx_(lh|rh)_', '', regex=True)
merged['severity'] = merged['true_label'].map({1: 'high', 0: 'low'})
merged = merged[~merged['area'].isin(EXCLUDE_AREAS)]

saliency_sub = saliency_sub.copy()
saliency_sub['area_full'] = saliency_sub['region_id'].map(idx_to_region)
saliency_sub = saliency_sub.dropna(subset=['area_full'])
extracted = saliency_sub['area_full'].str.extract(r'^(?P<area>.+)\s\((?P<hemi_letter>[LR])\)$')
saliency_sub['area'] = extracted['area']
saliency_sub['hemi'] = extracted['hemi_letter'].map(hemi_letter_map)
saliency_sub['severity'] = saliency_sub['true_label'].map({1: 'high', 0: 'low'})
saliency_sub = saliency_sub[~saliency_sub['area'].isin(EXCLUDE_AREAS)]

cortical_areas = merged[['area']].drop_duplicates()
cortical_areas = pd.merge(cortical_areas, lobe_info, on='area', how='left')
cortical_areas['lobe'] = cortical_areas['lobe'].fillna('unknown')

subcortical_areas = saliency_sub[['area']].drop_duplicates()
subcortical_areas['lobe'] = 'subcortical'

area_meta = pd.concat([cortical_areas, subcortical_areas], ignore_index=True).drop_duplicates('area')
area_meta['lobe'] = pd.Categorical(area_meta['lobe'], categories=lobe_order, ordered=True)
area_meta = area_meta.sort_values(by=['lobe', 'area']).reset_index(drop=True)

x_order = area_meta['area'].tolist()
x_lobes = area_meta['lobe'].astype(str).tolist()
is_subcortical = (area_meta['lobe'] == 'subcortical').values
n = len(x_order)
x_pos = np.arange(n) + 0.5

boundaries = []
start = 0
for i in range(1, n + 1):
    if i == n or x_lobes[i] != x_lobes[start]:
        boundaries.append((x_lobes[start], start, i - 1))
        start = i

agg = merged.groupby(['area', 'hemi', 'severity'], observed=True)[VALUE_COL].mean().reset_index()
wide = agg.pivot_table(index='area', columns=['hemi', 'severity'], values=VALUE_COL)
wide.columns = [f'{h}{s}average' for h, s in wide.columns]
wide = wide.reset_index()

agg_sub = saliency_sub.groupby(['area', 'hemi', 'severity'], observed=True)[VALUE_COL].mean().reset_index()
wide_sub = agg_sub.pivot_table(index='area', columns=['hemi', 'severity'], values=VALUE_COL)
wide_sub.columns = [f'{h}{s}average' for h, s in wide_sub.columns]
wide_sub = wide_sub.reset_index()

wide_all = pd.concat([wide, wide_sub], ignore_index=True)
for col in ['lhhighaverage', 'lhlowaverage', 'rhhighaverage', 'rhlowaverage']:
    if col not in wide_all.columns:
        wide_all[col] = pd.NA

wide_all = wide_all.set_index('area').reindex(x_order).reset_index()
lhhigh, lhlow = wide_all['lhhighaverage'], wide_all['lhlowaverage']
rhhigh, rhlow = wide_all['rhhighaverage'], wide_all['rhlowaverage']

cortical_avg = avg_by_fold(merged, VALUE_COL)
subcortical_avg = avg_by_fold(saliency_sub, VALUE_COL)
all_avg = pd.concat([cortical_avg, subcortical_avg], ignore_index=True)

folds = sorted(all_avg['fold'].unique())
hemis = ['lh', 'rh']

rows, row_labels = [], []
for hemi in hemis:
    for fold in folds:
        sub = all_avg[(all_avg['hemi'] == hemi) & (all_avg['fold'] == fold)]
        rows.append(sub.set_index('area')['avg'].reindex(x_order))
        row_labels.append(FOLD_SITE_NAMES.get(fold, f'Fold {fold}'))

matrix = pd.DataFrame(rows, index=row_labels)[x_order]  # columns already in x_order
matrix_scaled = matrix / VALUE_SCALE

flat_vals = matrix_scaled.values.flatten()
flat_vals = flat_vals[~np.isnan(flat_vals)]
abs_cap = np.nanpercentile(np.abs(flat_vals), 98) if len(flat_vals) else 1
vmin, vmax = -abs_cap, abs_cap

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(44, 32), sharex=True,
    gridspec_kw={'height_ratios': [1, 0.8], 'hspace': 0.55}
)

for i, (lobe_name, s_idx, e_idx) in enumerate(boundaries):
    if i % 2 == 0:
        ax1.axvspan(s_idx, e_idx + 1, color='gray', alpha=0.05, zorder=0)
    mid = (s_idx + e_idx + 1) / 2
    ax1.text(mid, 1.01, lobe_name.capitalize(), transform=ax1.get_xaxis_transform(),
              ha='center', va='bottom', fontsize=24, fontweight='bold', color='#555555')

ax1.vlines(x=x_pos, ymin=lhhigh, ymax=lhlow, color='grey', alpha=1)
ax1.scatter(x_pos, lhhigh, s=100, color='blue', alpha=0.6, label='LH high severity')
ax1.scatter(x_pos, lhlow, s=100, color='green', alpha=0.4, label='LH low severity')
ax1.vlines(x=x_pos, ymin=rhhigh, ymax=rhlow, color='grey', alpha=1)
ax1.scatter(x_pos, rhhigh, s=100, color='red', alpha=1, label='RH high severity')
ax1.scatter(x_pos, rhlow, s=100, color='orange', alpha=0.4, label='RH low severity')

ax1.set_ylabel('Average Saliency', fontsize=28, fontweight='bold')
ax1.tick_params(axis='y', labelsize=24)
ax1.yaxis.set_major_formatter(FuncFormatter(sci_fmt))
ax1.legend(prop={'size': 22}, loc='lower right', frameon=True,
           edgecolor='#CCCCCC', framealpha=0.95)
ax1.set_xlim(0, n)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_order, rotation=90, fontsize=24)
ax1.tick_params(axis='x', labelbottom=True)
for tick, is_sub in zip(ax1.get_xticklabels(), is_subcortical):
    if is_sub:
        tick.set_fontweight('bold')
ax1.invert_yaxis()

sns.heatmap(matrix_scaled, ax=ax2, cmap='RdBu_r', vmin=vmin, vmax=vmax,
            center=0, linewidths=0.6, linecolor='#AAAAAA', cbar=False)   

for lobe_name, s_idx, e_idx in boundaries:
    ax2.axvline(s_idx, color='black', linewidth=1.0)
ax2.axvline(n, color='black', linewidth=1.0)

ax2.set_xlabel('Destrieux-based cortical / ASEG-based subcortical parcellation',
               fontsize=28, fontweight='bold')
ax2.set_ylabel('  RH                           LH', fontsize=28, fontweight='bold')
ax2.tick_params(axis='x', labelsize=24, rotation=90)
ax2.tick_params(axis='y', labelsize=24, rotation=0)
ax2.set_xticks(np.arange(n + 1), minor=True)
ax2.set_yticks(np.arange(len(row_labels) + 1), minor=True)
ax2.grid(which='minor', color='#AAAAAA', linewidth=0.6)
ax2.tick_params(which='minor', bottom=False, left=False) 

for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

for tick, is_sub in zip(ax2.get_xticklabels(), is_subcortical):
    if is_sub:
        tick.set_fontweight('bold')

fig.subplots_adjust(right=0.92)

pos2 = ax2.get_position()
cbar_ax = fig.add_axes([0.935, pos2.y0, 0.006, pos2.height]) 

sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: sci_fmt(x * VALUE_SCALE, pos))
)
cbar.set_label('Average Saliency', fontsize=24, fontweight='bold', labelpad=15)
cbar.ax.tick_params(labelsize=20, width=1, length=4)

plt.show()